<a href="https://colab.research.google.com/github/lankipolo123/roadfixqc/blob/main/roadblocks_yolo11n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# A100 GPU - 90%+ ACCURACY - ROADBLOCKS DETECTION (YOLOv11n + LIGHT BLUR)
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")

# INSTALL
!pip install ultralytics roboflow opencv-python

import os
import shutil
import cv2
import numpy as np
from ultralytics import YOLO
import torch
from google.colab import files
import yaml

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================================
# DOWNLOAD ROADBLOCKS DATASET
# ============================================================================

print("\n📥 Downloading Roadblocks Dataset...")

!pip install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="SzttdelfmuWaCwAz2N5u")
project = rf.workspace("dequillaprojects").project("roadblocks-qifzg")
version = project.version(2)
dataset = version.download("yolov11")

dataset_path = dataset.location
print(f"✅ Dataset downloaded to: {dataset_path}")

# ============================================================================
# CHECK FOLDER STRUCTURE & CLASS NAMES
# ============================================================================

print("\n📁 Checking dataset structure...")
for item in os.listdir(dataset_path):
    print(f"  - {item}")

# Read data.yaml to find class names
yaml_path = f"{dataset_path}/data.yaml"
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("\n📋 Classes in dataset:")
for idx, class_name in enumerate(data_config['names']):
    print(f"  {idx}: {class_name}")

# Find Fallen_Tree class index
fallen_tree_idx = None
for idx, class_name in enumerate(data_config['names']):
    if 'fallen' in class_name.lower() or 'tree' in class_name.lower():
        fallen_tree_idx = idx
        print(f"\n🌳 Found Fallen_Tree at index: {idx} - '{class_name}'")
        break

if fallen_tree_idx is None:
    print("\n⚠️ Fallen_Tree class not found! Proceeding without class-specific blur.")

# Determine folder structure
if os.path.exists(f"{dataset_path}/train/images"):
    train_folder = "train"
    val_folder = "valid"
elif os.path.exists(f"{dataset_path}/train"):
    train_folder = "train"
    val_folder = "val"
else:
    print("\n❌ Unknown structure:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        if level < 2:
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
    raise Exception("Update folder names manually")

train_images_path = f"{dataset_path}/{train_folder}/images"
train_labels_path = f"{dataset_path}/{train_folder}/labels"
val_images_path = f"{dataset_path}/{val_folder}/images"
val_labels_path = f"{dataset_path}/{val_folder}/labels"

train_images = len(os.listdir(train_images_path))
val_images = len(os.listdir(val_images_path))
print(f"\n✅ Training images: {train_images}")
print(f"✅ Validation images: {val_images}")

# ============================================================================
# LIGHT BLUR FOR FALLEN_TREE CLASS
# ============================================================================

if fallen_tree_idx is not None:
    print(f"\n🌳 Applying LIGHT BLUR to Fallen_Tree class (index {fallen_tree_idx})...")

    def apply_light_blur_to_class(images_path, labels_path, class_id):
        """Apply LIGHT Gaussian blur to images containing specific class"""

        image_files = [f for f in os.listdir(images_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
        blurred_count = 0

        for img_file in image_files:
            label_file = img_file.rsplit('.', 1)[0] + '.txt'
            label_path = os.path.join(labels_path, label_file)

            if not os.path.exists(label_path):
                continue

            # Check if target class exists
            with open(label_path, 'r') as f:
                labels = f.readlines()

            has_target_class = any(line.strip().startswith(f'{class_id} ') for line in labels)

            if has_target_class:
                img_path = os.path.join(images_path, img_file)
                img = cv2.imread(img_path)

                if img is None:
                    continue

                # Apply LIGHT Gaussian Blur (3x3 kernel, sigma=0.5)
                blurred = cv2.GaussianBlur(img, (3, 3), 0.5)

                # Save blurred image
                cv2.imwrite(img_path, blurred)
                blurred_count += 1

        return blurred_count

    # Apply to training set
    train_blurred = apply_light_blur_to_class(train_images_path, train_labels_path, fallen_tree_idx)
    print(f"✅ Applied light blur to {train_blurred} training images with Fallen_Tree")

    # Apply to validation set
    val_blurred = apply_light_blur_to_class(val_images_path, val_labels_path, fallen_tree_idx)
    print(f"✅ Applied light blur to {val_blurred} validation images with Fallen_Tree")
else:
    print("\n⚠️ Skipping class-specific blur (Fallen_Tree not found)")

# ============================================================================
# A100 OPTIMIZED TRAINING - YOLOv11n - 90%+ TARGET
# ============================================================================

print("\n🚀 Starting A100 OPTIMIZED Training with YOLOv11n...")

drive_project_path = '/content/drive/MyDrive/roadblocks_training_a100'
os.makedirs(drive_project_path, exist_ok=True)

# YOLOv11n - NANO MODEL (as you originally wanted)
model = YOLO('yolo11n.pt')

results = model.train(
    data=f'{dataset_path}/data.yaml',
    epochs=250,              # More epochs for nano model to reach 90%
    imgsz=1024,
    batch=32,
    lr0=0.001,
    lrf=0.01,
    optimizer='AdamW',
    cos_lr=True,
    patience=60,
    save_period=10,

    # AGGRESSIVE AUGMENTATION
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.15,
    scale=0.5,
    shear=5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.15,

    # LOSS WEIGHTS
    box=8.0,
    cls=0.5,
    dfl=1.5,

    name='yolov11n_roadblocks_a100_v1',
    project=drive_project_path,
    exist_ok=True,
    device=0,
    workers=8,
    amp=True,
    cache=True
)

print("✅ Initial training complete!")

# ============================================================================
# VALIDATION & AUTO FINE-TUNING
# ============================================================================

val_results = model.val()
current_map50 = val_results.box.map50

print(f"\n📊 Current mAP50: {current_map50:.4f} ({current_map50*100:.1f}%)")
print(f"📊 mAP50-95: {val_results.box.map:.4f}")
print(f"📊 Precision: {val_results.box.mp:.4f}")
print(f"📊 Recall: {val_results.box.mr:.4f}")

# If not 90%, auto fine-tune
if current_map50 < 0.90:
    print(f"\n🔧 Current: {current_map50*100:.1f}% - AUTO FINE-TUNING for 90%+...")

    best_model_path = f'{drive_project_path}/yolov11n_roadblocks_a100_v1/weights/best.pt'
    fine_tune_model = YOLO(best_model_path)

    fine_tune_results = fine_tune_model.train(
        data=f'{dataset_path}/data.yaml',
        epochs=100,
        imgsz=1280,
        batch=24,
        lr0=0.0001,
        optimizer='AdamW',
        cos_lr=True,
        patience=30,

        # Reduced augmentation for fine-tuning
        mosaic=0.3,
        mixup=0.05,
        scale=0.5,

        box=8.0,

        name='yolov11n_roadblocks_finetune_v1',
        project=drive_project_path,
        exist_ok=True,
        device=0,
        workers=8,
        amp=True,
        cache=True
    )

    val_results = fine_tune_model.val()
    current_map50 = val_results.box.map50
    print(f"\n✅ Fine-tuned mAP50: {current_map50:.4f} ({current_map50*100:.1f}%)")

    best_model_path = f'{drive_project_path}/yolov11n_roadblocks_finetune_v1/weights/best.pt'

# ============================================================================
# FINAL RESULTS
# ============================================================================

final_accuracy = current_map50 * 100

print("\n" + "="*60)
print("🎯 ROADBLOCKS DETECTION - YOLOv11n - A100 OPTIMIZED")
print("="*60)
print(f"✅ ACCURACY (mAP50): {final_accuracy:.2f}%")
print(f"📊 mAP50-95: {val_results.box.map:.4f}")
print(f"📊 Precision: {val_results.box.mp:.4f}")
print(f"📊 Recall: {val_results.box.mr:.4f}")
print(f"📊 Training images: {train_images}")
print(f"📊 Validation images: {val_images}")

if fallen_tree_idx is not None:
    print(f"🌳 Fallen_Tree light blur: ✅ Applied (3x3, sigma=0.5)")

if final_accuracy >= 90:
    print(f"\n🎉 TARGET ACHIEVED: {final_accuracy:.1f}% >= 90%")
else:
    print(f"\n⚠️ Target not reached: {final_accuracy:.1f}% < 90%")

# ============================================================================
# SAVE TO DRIVE
# ============================================================================

final_model_name = f"BEST_Roadblocks_YOLOv11n_{final_accuracy:.1f}percent.pt"
shutil.copy(best_model_path, f'/content/drive/MyDrive/{final_model_name}')
print(f"\n💾 SAVED TO DRIVE: MyDrive/{final_model_name}")

# ============================================================================
# EXPORT FORMATS
# ============================================================================

print("\n📦 Exporting model formats...")
final_model = YOLO(best_model_path)
final_model.export(format='onnx')
final_model.export(format='torchscript')
print("✅ Exported: ONNX, TorchScript")

# ============================================================================
# CREATE DOWNLOAD PACKAGE
# ============================================================================

print("\n📥 Creating download package...")

download_folder = '/content/ROADBLOCKS_DOWNLOAD'
os.makedirs(download_folder, exist_ok=True)

# Copy best model
shutil.copy(best_model_path, f'{download_folder}/{final_model_name}')

# Copy training results
training_folder = f'{drive_project_path}/yolov11n_roadblocks_a100_v1'
if not os.path.exists(training_folder):
    training_folder = f'{drive_project_path}/yolov11n_roadblocks_finetune_v1'

result_files = [
    'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
    'F1_curve.png', 'PR_curve.png', 'P_curve.png', 'R_curve.png',
    'labels.jpg', 'labels_correlogram.jpg'
]

for file in result_files:
    src = f'{training_folder}/{file}'
    if os.path.exists(src):
        shutil.copy(src, f'{download_folder}/{file}')

# Copy exported formats
for file in os.listdir('.'):
    if file.endswith(('.onnx', '.torchscript')):
        shutil.copy(file, f'{download_folder}/{file}')

# Create summary file
with open(f'{download_folder}/RESULTS_SUMMARY.txt', 'w') as f:
    f.write("="*60 + "\n")
    f.write("ROADBLOCKS DETECTION - YOLOv11n - A100 OPTIMIZED\n")
    f.write("="*60 + "\n\n")
    f.write(f"Model: YOLOv11n (NANO)\n")
    f.write(f"GPU: {torch.cuda.get_device_name(0)}\n\n")
    f.write(f"Final Accuracy (mAP50): {final_accuracy:.2f}%\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall: {val_results.box.mr:.4f}\n\n")
    f.write(f"Training Images: {train_images}\n")
    f.write(f"Validation Images: {val_images}\n\n")
    f.write("Optimizations:\n")
    f.write("- A100 GPU\n")
    f.write("- YOLOv11n (nano model)\n")
    f.write("- Image size: 1024px (fine-tune: 1280px)\n")
    f.write("- Aggressive augmentation\n")
    if fallen_tree_idx is not None:
        f.write(f"- Light blur on Fallen_Tree class (3x3, sigma=0.5)\n")
    f.write("- Auto fine-tuning if < 90%\n\n")
    f.write("Classes:\n")
    for idx, class_name in enumerate(data_config['names']):
        f.write(f"  {idx}: {class_name}\n")

# Create ZIP
shutil.make_archive('/content/Roadblocks_YOLOv11n_Complete', 'zip', download_folder)

print("\n" + "="*60)
print("📥 DOWNLOADING COMPLETE PACKAGE...")
print("="*60)
files.download('/content/Roadblocks_YOLOv11n_Complete.zip')

print("\n🎉 DONE! YOLOv11n model ready!")

Mounted at /content/drive
✅ Google Drive mounted!
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 105.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.conf


Extracting Dataset Version Zip to RoadBlocks-2 in yolov11:: 100%|██████████| 38586/38586 [00:06<00:00, 6198.57it/s]


✅ Dataset downloaded to: /content/RoadBlocks-2

📁 Checking dataset structure...
  - README.dataset.txt
  - train
  - data.yaml
  - test
  - valid
  - README.roboflow.txt

📋 Classes in dataset:
  0: Fallen_Tree
  1: Road_Barrier
  2: Stable_Tree
  3: Tires
  4: Tires_with_rim
  5: Traffic_Cones

🌳 Found Fallen_Tree at index: 0 - 'Fallen_Tree'

✅ Training images: 13517
✅ Validation images: 2956

🌳 Applying LIGHT BLUR to Fallen_Tree class (index 0)...
✅ Applied light blur to 5714 training images with Fallen_Tree
✅ Applied light blur to 1223 validation images with Fallen_Tree

🚀 Starting A100 OPTIMIZED Training with YOLOv11n...
Ultralytics 8.3.213 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=8.0, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/

NameError: name 'best_model_path' is not defined